In [86]:
pip install -U albumentations

Note: you may need to restart the kernel to use updated packages.


In [87]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model, layers, regularizers
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import albumentations as A


In [88]:

# ==========================================================
# Augmentation Pipeline
# ==========================================================

transform = A.Compose([

    # ----------------------------
    # Camera angle simulation
    # ----------------------------
    A.Affine(
        rotate=(-5, 5),
        translate_percent=(-0.05, 0.05),
        scale=(0.90, 1.10),
        shear=(-5, 5),
        border_mode=cv2.BORDER_REPLICATE,
        p=0.80
    ),

    A.Perspective(
        scale=(0.02, 0.08),
        keep_size=True,
        p=0.50
    ),

    # ----------------------------
    # Brightness / Contrast
    # ----------------------------
    A.RandomBrightnessContrast(
        brightness_limit=0.25,
        contrast_limit=0.25,
        p=0.60
    ),

    # ----------------------------
    # Gamma
    # ----------------------------
    A.RandomGamma(
        gamma_limit=(80,120),
        p=0.30
    ),

    # ----------------------------
    # CLAHE
    # ----------------------------
    A.CLAHE(
        clip_limit=3,
        tile_grid_size=(8,8),
        p=0.30
    ),




   

    # ----------------------------
    # Noise
    # ----------------------------
    A.OneOf([

        A.GaussNoise(
            std_range=(0.02,0.08),
            p=1
        ),

        A.ISONoise(
            color_shift=(0.01,0.03),
            intensity=(0.1,0.3),
            p=1
        ),

    ], p=0.25),

    # ----------------------------
    # Slight Color Changes
    # ----------------------------
    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=10,
        val_shift_limit=10,
        p=0.20
    ),

], p=1.0)


# ==========================================================
# TensorFlow Wrapper
# ==========================================================

def albumentations(image):

    # NOTE: this function runs inside tf.numpy_function, which -- unlike
    # tf.py_function -- passes plain NumPy arrays into the callback, not
    # eager tensors. So `image` here is already a np.ndarray; calling
    # image.numpy() (as the original code did) raises
    # "AttributeError: 'numpy.ndarray' object has no attribute 'numpy'".
    # (tf.py_function would give you an eager tensor with .numpy(); this is
    # tf.numpy_function specifically, so no conversion call is needed here.)

    # image already arrives in [0,255] float32 (see load_image below) --
    # EfficientNetB0 has its own internal Rescaling/Normalization layer and
    # expects raw [0,255] pixel input, so we must NOT divide by 255 anywhere
    # in this pipeline. We only clip+cast to uint8 here because albumentations
    # needs uint8, and clipping (not a bare astype) avoids uint8 wraparound
    # from any out-of-range values (e.g. resize interpolation overshoot).
    image = np.clip(image, 0, 255).astype(np.uint8)

    image = transform(image=image)["image"]

    # cast back to float32, still in [0,255] -- no /255 here
    image = image.astype(np.float32)

    return image


def tf_augment(image, labels, weights):

    image = tf.numpy_function(
        albumentations,
        [image],
        tf.float32
    )

    image.set_shape((96,320,3))

    # weights pass through untouched -- augmentation changes the image,
    # not which class-frequency bucket a label belongs to.
    return image, labels, weights

In [89]:
IMG_HEIGHT = 96
IMG_WIDTH = 320
CHANNELS = 3

BATCH_SIZE = 32
EPOCHS = 60

AUTOTUNE = tf.data.AUTOTUNE

In [90]:
def compute_inverse_freq_weights(digit_series, num_classes=10):
    """
    Standard inverse-frequency class weighting:
    weight[c] = total_count / (num_classes * count[c])
    Classes that never appear get count clamped to 1 to avoid div-by-zero
    (their weight ends up large, but they contribute 0 samples anyway).
    """
    counts = digit_series.value_counts().reindex(range(num_classes), fill_value=0).values.astype(np.float32)
    counts = np.where(counts == 0, 1, counts)
    total = counts.sum()
    return total / (num_classes * counts)


def create_dataset(csv_path, image_folder, shuffle=True, use_class_weights=False):

    df = pd.read_csv(csv_path, dtype={"label": str})

    df["label"] = df["label"].str.zfill(5)

    image_paths = [
        os.path.join(image_folder, img)
        for img in df["image"]
    ]

    digit_series = [df["label"].str[i].astype(int) for i in range(5)]
    digit_arrays = tuple(s.values for s in digit_series)

    # Class weights are computed from and applied ONLY to the training set.
    # Validation/test must reflect the true, unweighted class distribution --
    # weighting them would make val_loss/val_accuracy misleading rather than
    # informative.
    if use_class_weights:
        weight_arrays = []
        for s in digit_series:
            class_w = compute_inverse_freq_weights(s)
            weight_arrays.append(s.map(lambda v, cw=class_w: cw[v]).astype(np.float32).values)
        weight_arrays = tuple(weight_arrays)
    else:
        weight_arrays = tuple(np.ones(len(df), dtype=np.float32) for _ in range(5))

    dataset = tf.data.Dataset.from_tensor_slices((
        image_paths,
        digit_arrays,
        weight_arrays,
    ))

    def load_image(path, digits, weights):

        image = tf.io.read_file(path)

        image = tf.image.decode_png(image, channels=3)

        image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))

        # Raw [0,255] float32 -- ResNet50's preprocess_input is applied
        # inside the model itself (see the Lambda layer), so this pipeline
        # must NOT rescale or normalize.
        image = tf.cast(image, tf.float32)

        labels = (digits[0], digits[1], digits[2], digits[3], digits[4])
        sample_weights = (weights[0], weights[1], weights[2], weights[3], weights[4])

        return image, labels, sample_weights

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:

        dataset = dataset.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

        dataset = dataset.map(
            tf_augment,
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


### Sanity check before training: is `digit1` actually a meaningful target?

The earlier training run showed `digit1_accuracy: 1.0000` while `digit5_accuracy` was only 0.20. Before treating digit1 as "solved", check whether it's simply a near-constant label in your dataset (e.g. always `0` for smaller meter readings) — if so, 100% accuracy there is trivial, not evidence the model learned anything meaningful.

In [91]:
check_df = pd.read_csv("../data_set_generator_for_cnns_only/data_set/train.csv", dtype={"label": str})
check_df["label"] = check_df["label"].str.zfill(5)

for i in range(5):
    print(f"digit{i+1} value counts:")
    print(check_df["label"].str[i].value_counts(normalize=True).sort_index())
    print()

digit1 value counts:
label
0    0.986637
1    0.011136
8    0.002227
Name: proportion, dtype: float64

digit2 value counts:
label
0    0.832962
1    0.042316
2    0.044543
3    0.026726
4    0.006682
5    0.020045
6    0.008909
7    0.006682
8    0.008909
9    0.002227
Name: proportion, dtype: float64

digit3 value counts:
label
0    0.652561
1    0.093541
2    0.028953
3    0.064588
4    0.035635
5    0.033408
6    0.015590
7    0.011136
8    0.051225
9    0.013363
Name: proportion, dtype: float64

digit4 value counts:
label
0    0.282851
1    0.296214
2    0.113586
3    0.055679
4    0.044543
5    0.028953
6    0.033408
7    0.026726
8    0.071269
9    0.046771
Name: proportion, dtype: float64

digit5 value counts:
label
0    0.075724
1    0.118040
2    0.111359
3    0.075724
4    0.104677
5    0.066815
6    0.140312
7    0.115813
8    0.080178
9    0.111359
Name: proportion, dtype: float64



In [92]:
train_ds = create_dataset(
    "../data_set_generator_for_cnns_only/data_set/train.csv",
    "../data_set_generator_for_cnns_only/data_set/train",
    shuffle=True,
    use_class_weights=True   # only the training set gets reweighted
)

valid_ds = create_dataset(
    "../data_set_generator_for_cnns_only/data_set/valid.csv",
    "../data_set_generator_for_cnns_only/data_set/valid",
    shuffle=False,
    use_class_weights=False  # validation must reflect the true distribution
)

test_ds = create_dataset(
    "../data_set_generator_for_cnns_only/data_set/test.csv",
    "../data_set_generator_for_cnns_only/data_set/test",
    shuffle=False,
    use_class_weights=False
)


### Note on `base_model.trainable = False`

With the preprocessing bug fixed, re-run training first before changing anything else. If digit2-5 accuracy is still weak after that fix, the next thing to try is unfreezing the top block or two of EfficientNetB0 (fine-tuning) rather than using it purely as a frozen feature extractor — frozen ImageNet features tend to underperform when the target domain (small synthetic digit strips) looks very different from natural photos, since there's no way for the frozen filters to adapt. Fine-tune with a low learning rate (e.g. 1e-5) if you go this route, to avoid destroying the pretrained weights.

In [93]:
# ResNet50 backbone. Note: preprocess_input is applied INSIDE the model
# graph (see next cell), not here and not in the tf.data pipeline. This is
# deliberate -- it's what guarantees predict_meter() and training can never
# drift out of sync on preprocessing again, the way EfficientNet/255 did.

base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Phase 1: freeze the whole backbone and train only the new head.
# We deliberately do NOT fine-tune yet -- training a randomly-initialized
# head end-to-end with an unfrozen pretrained backbone from step 1 risks
# large early gradients wrecking the pretrained weights. Unfreezing happens
# in a later cell, after the head has already learned something sensible.
base_model.trainable = False


In [94]:
L2 = 1e-4

inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))

x = layers.Lambda(preprocess_input, name="resnet_preprocess")(inputs)

feat = base_model(x)  # verified shape: (None, 3, 10, 2048) at (96,320) input

# Global context: same for every digit, captures overall lighting/contrast.
global_pooled = layers.GlobalAveragePooling2D(name="global_pool")(feat)

# Each digit gets its OWN 2-column slice of the feature map (10 columns / 5
# digits = 2 columns each, verified empirically above) instead of every
# digit head seeing the exact same globally-pooled vector. This bakes in a
# prior we already know is true -- fixed left-to-right digit positions --
# instead of asking the model to learn it from ~256 images. Concatenating
# with global_pooled keeps some whole-image context as a hedge against
# slight column misalignment.
def digit_head(col_index, name):
    local = layers.Lambda(
        lambda t, i=col_index: t[:, :, i*2:(i+1)*2, :],
        name=f"{name}_slice"
    )(feat)
    local_pooled = layers.GlobalAveragePooling2D(name=f"{name}_local_pool")(local)

    merged = layers.Concatenate(name=f"{name}_merge")([global_pooled, local_pooled])

    h = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2))(merged)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(0.3)(h)
    return layers.Dense(10, activation="softmax", name=name)(h)

digit1 = digit_head(0, "digit1")
digit2 = digit_head(1, "digit2")
digit3 = digit_head(2, "digit3")
digit4 = digit_head(3, "digit4")
digit5 = digit_head(4, "digit5")

model = keras.Model(inputs, [digit1, digit2, digit3, digit4, digit5])


In [95]:
# NOTE: label_smoothing is NOT a valid argument for
# keras.losses.SparseCategoricalCrossentropy in current Keras (verified by
# testing -- it raises TypeError). Getting label smoothing would require
# switching to one-hot labels + CategoricalCrossentropy, which is more
# invasive than it's worth here, so using it plain instead.
loss_fn = keras.losses.SparseCategoricalCrossentropy()

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={
        "digit1": loss_fn,
        "digit2": loss_fn,
        "digit3": loss_fn,
        "digit4": loss_fn,
        "digit5": loss_fn,
    },
    metrics={
        "digit1": "accuracy",
        "digit2": "accuracy",
        "digit3": "accuracy",
        "digit4": "accuracy",
        "digit5": "accuracy",
    }
)


### Before running this: a real constraint, not a formality

Your training set is ~256 images. `digit1` is ~99% one value (trivial), but `digit4` and `digit5` are close to uniform across all 10 classes -- the hardest kind of target, and the one that needs the most examples per class. These architecture/training changes (correct ResNet50 preprocessing, fine-tuning, regularization, exact-match metric) fix real bugs and give the model its best shot, but none of them manufacture data that isn't there. If `digit4`/`digit5` exact-match accuracy is still weak after this, the next lever to pull is **more labeled images**, particularly ones covering underrepresented digit4/digit5 combinations -- not further architecture tweaks.

In [96]:
# Phase 1: warmup -- backbone frozen, only the new head trains.
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ModelCheckpoint("best_phase1.weights.h5", monitor="val_loss", save_best_only=True),
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Epoch 1/60


d:\internship\.venv(app_)\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - digit1_accuracy: 0.1183 - digit1_loss: 0.5580 - digit2_accuracy: 0.1138 - digit2_loss: 3.2952 - digit3_accuracy: 0.1853 - digit3_loss: 3.0415 - digit4_accuracy: 0.2031 - digit4_loss: 3.0072 - digit5_accuracy: 0.1629 - digit5_loss: 2.9168 - loss: 12.9431

15/15 ━━━━━━━━━━━━━━━━━━━━ 18s 651ms/step - digit1_accuracy: 0.1203 - digit1_loss: 0.5360 - digit2_accuracy: 0.1136 - digit2_loss: 3.6492 - digit3_accuracy: 0.1849 - digit3_loss: 3.1386 - digit4_accuracy: 0.2027 - digit4_loss: 2.9423 - digit5_accuracy: 0.1626 - digit5_loss: 2.8600 - loss: 12.9533 - val_digit1_accuracy: 0.3898 - val_digit1_loss: 1.8551 - val_digit2_accuracy: 0.0339 - val_digit2_loss: 6.3856 - val_digit3_accuracy: 0.6102 - val_digit3_loss: 2.3679 - val_digit4_accuracy: 0.2881 - val_digit4_loss: 3.0902 - val_digit5_accuracy: 0.2542 - val_digit5_loss: 3.8819 - val_loss: 17.7182 - learning_rate: 0.0010
Epoch 2/60
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 476ms/step - digit1_accuracy: 0.1429 - digit1_loss: 0.6102 - digit2_accuracy: 0.1696 - digit2_loss: 1.9996 - digit3_accuracy: 0.3237 - digit3_loss: 2.0206 - digit4_accuracy: 0.3326 - digit4_loss: 2.0964 - digit5_accuracy: 0.2679 - digit5_loss: 2.3556 - loss: 9.2084

15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 658ms/step - digit1_accuracy: 0.1448 - digit1_loss: 0.5840 - digit2_accuracy: 0.1715 - digit2_loss: 1.8843 - digit3_accuracy: 0.3229 - digit3_loss: 1.9092 - digit4_accuracy: 0.3341 - digit4_loss: 2.0070 - digit5_accuracy: 0.2673 - digit5_loss: 2.3463 - loss: 9.1966 - val_digit1_accuracy: 0.3898 - val_digit1_loss: 1.8352 - val_digit2_accuracy: 0.0508 - val_digit2_loss: 5.4761 - val_digit3_accuracy: 0.6949 - val_digit3_loss: 1.8589 - val_digit4_accuracy: 0.3898 - val_digit4_loss: 2.2152 - val_digit5_accuracy: 0.2373 - val_digit5_loss: 2.9923 - val_loss: 14.4266 - learning_rate: 0.0010
Epoch 3/60
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - digit1_accuracy: 0.1339 - digit1_loss: 0.3571 - digit2_accuracy: 0.2232 - digit2_loss: 1.7435 - digit3_accuracy: 0.4286 - digit3_loss: 1.5609 - digit4_accuracy: 0.4152 - digit4_loss: 1.7052 - digit5_accuracy: 0.2902 - digit5_loss: 2.1683 - loss: 7.6625

...

15/15 ━━━━━━━━━━━━━━━━━━━━ 9s 624ms/step - digit1_accuracy: 0.9220 - digit1_loss: 0.0281 - digit2_accuracy: 0.6860 - digit2_loss: 0.6314 - digit3_accuracy: 0.8062 - digit3_loss: 0.5207 - digit4_accuracy: 0.7617 - digit4_loss: 0.6145 - digit5_accuracy: 0.6704 - digit5_loss: 1.1665 - loss: 3.0312 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3735 - val_digit2_accuracy: 0.6949 - val_digit2_loss: 1.3205 - val_digit3_accuracy: 0.6780 - val_digit3_loss: 1.0474 - val_digit4_accuracy: 0.6441 - val_digit4_loss: 1.4402 - val_digit5_accuracy: 0.4915 - val_digit5_loss: 1.5834 - val_loss: 5.9185 - learning_rate: 5.0000e-04
Epoch 32/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 492ms/step - digit1_accuracy: 0.9488 - digit1_loss: 0.0432 - digit2_accuracy: 0.7016 - digit2_loss: 0.3579 - digit3_accuracy: 0.8129 - digit3_loss: 0.4798 - digit4_accuracy: 0.7773 - digit4_loss: 0.6133 - digit5_accuracy: 0.6281 - digit5_loss: 1.1483 - loss: 2.7789 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3452 - val_digit2_

15/15 ━━━━━━━━━━━━━━━━━━━━ 9s 624ms/step - digit1_accuracy: 0.9443 - digit1_loss: 0.0213 - digit2_accuracy: 0.7416 - digit2_loss: 0.3838 - digit3_accuracy: 0.8129 - digit3_loss: 0.4355 - digit4_accuracy: 0.7884 - digit4_loss: 1.1943 - digit5_accuracy: 0.6771 - digit5_loss: 1.0625 - loss: 2.6805 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3269 - val_digit2_accuracy: 0.7288 - val_digit2_loss: 1.1283 - val_digit3_accuracy: 0.6271 - val_digit3_loss: 1.3406 - val_digit4_accuracy: 0.6780 - val_digit4_loss: 1.3851 - val_digit5_accuracy: 0.4068 - val_digit5_loss: 1.5388 - val_loss: 5.8576 - learning_rate: 5.0000e-04
Epoch 35/60
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - digit1_accuracy: 0.9554 - digit1_loss: 0.0204 - digit2_accuracy: 0.7411 - digit2_loss: 0.3871 - digit3_accuracy: 0.8013 - digit3_loss: 0.3832 - digit4_accuracy: 0.8013 - digit4_loss: 0.5697 - digit5_accuracy: 0.6674 - digit5_loss: 1.0309 - loss: 2.5398

15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 667ms/step - digit1_accuracy: 0.9555 - digit1_loss: 0.0193 - digit2_accuracy: 0.7416 - digit2_loss: 0.3692 - digit3_accuracy: 0.8018 - digit3_loss: 0.3663 - digit4_accuracy: 0.7996 - digit4_loss: 0.5741 - digit5_accuracy: 0.6659 - digit5_loss: 1.1992 - loss: 2.5444 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3236 - val_digit2_accuracy: 0.7288 - val_digit2_loss: 1.0977 - val_digit3_accuracy: 0.6780 - val_digit3_loss: 1.0850 - val_digit4_accuracy: 0.6271 - val_digit4_loss: 1.4508 - val_digit5_accuracy: 0.4576 - val_digit5_loss: 1.5115 - val_loss: 5.6081 - learning_rate: 5.0000e-04
Epoch 36/60
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - digit1_accuracy: 0.9621 - digit1_loss: 0.0275 - digit2_accuracy: 0.7500 - digit2_loss: 0.4767 - digit3_accuracy: 0.7991 - digit3_loss: 0.4978 - digit4_accuracy: 0.7902 - digit4_loss: 0.5888 - digit5_accuracy: 0.6496 - digit5_loss: 1.0370 - loss: 2.7764

15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 661ms/step - digit1_accuracy: 0.9621 - digit1_loss: 0.0259 - digit2_accuracy: 0.7506 - digit2_loss: 0.4519 - digit3_accuracy: 0.7973 - digit3_loss: 0.7217 - digit4_accuracy: 0.7884 - digit4_loss: 1.1037 - digit5_accuracy: 0.6481 - digit5_loss: 1.1796 - loss: 2.8050 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3075 - val_digit2_accuracy: 0.7119 - val_digit2_loss: 1.1254 - val_digit3_accuracy: 0.7627 - val_digit3_loss: 0.9190 - val_digit4_accuracy: 0.6780 - val_digit4_loss: 1.3813 - val_digit5_accuracy: 0.4407 - val_digit5_loss: 1.4916 - val_loss: 5.3755 - learning_rate: 5.0000e-04
Epoch 37/60
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 453ms/step - digit1_accuracy: 0.9397 - digit1_loss: 0.0253 - digit2_accuracy: 0.7478 - digit2_loss: 0.3256 - digit3_accuracy: 0.8304 - digit3_loss: 0.4308 - digit4_accuracy: 0.7835 - digit4_loss: 0.5265 - digit5_accuracy: 0.7031 - digit5_loss: 0.9562 - loss: 2.4131

15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 644ms/step - digit1_accuracy: 0.9399 - digit1_loss: 0.0237 - digit2_accuracy: 0.7483 - digit2_loss: 0.3087 - digit3_accuracy: 0.8307 - digit3_loss: 0.4108 - digit4_accuracy: 0.7817 - digit4_loss: 0.6211 - digit5_accuracy: 0.7038 - digit5_loss: 0.9949 - loss: 2.4162 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3015 - val_digit2_accuracy: 0.7119 - val_digit2_loss: 1.1054 - val_digit3_accuracy: 0.7627 - val_digit3_loss: 0.9293 - val_digit4_accuracy: 0.6949 - val_digit4_loss: 1.3581 - val_digit5_accuracy: 0.4746 - val_digit5_loss: 1.4351 - val_loss: 5.2789 - learning_rate: 5.0000e-04
Epoch 38/60
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 489ms/step - digit1_accuracy: 0.9599 - digit1_loss: 0.0183 - digit2_accuracy: 0.7506 - digit2_loss: 0.2797 - digit3_accuracy: 0.8174 - digit3_loss: 0.4049 - digit4_accuracy: 0.7751 - digit4_loss: 0.5465 - digit5_accuracy: 0.6503 - digit5_loss: 1.1095 - loss: 2.4686 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.2963 - val_digit2

### Phase 2: fine-tune the top of ResNet50

Now that the head has learned something sensible with a frozen backbone, unfreeze the last residual block (`conv5_block*`) and continue training at a much lower learning rate. This lets the highest-level features adapt to your digit strips specifically, instead of staying stuck as generic ImageNet features -- without a low LR here, you risk destroying the pretrained weights in a couple of batches.

In [97]:
# Unfreeze only the last residual block, not the whole backbone -- the
# early layers encode generic edge/texture detectors that are still useful
# and don't need to change; only the highest-level, most task-specific
# layers benefit from adapting to your domain.
base_model.trainable = True

for layer in base_model.layers:
    if not layer.name.startswith("conv5_block"):
        layer.trainable = False

# Recompile is required after changing .trainable -- Keras won't pick up
# the change otherwise. Note the much lower LR: this is fine-tuning, not
# training from scratch.
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss={
        "digit1": loss_fn,
        "digit2": loss_fn,
        "digit3": loss_fn,
        "digit4": loss_fn,
        "digit5": loss_fn,
    },
    metrics={
        "digit1": "accuracy",
        "digit2": "accuracy",
        "digit3": "accuracy",
        "digit4": "accuracy",
        "digit5": "accuracy",
    }
)

finetune_callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
    ModelCheckpoint("best_finetuned.weights.h5", monitor="val_loss", save_best_only=True),
]



In [98]:
history_finetune = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=40,
    callbacks=finetune_callbacks
)


Epoch 1/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 660ms/step - digit1_accuracy: 0.9087 - digit1_loss: 0.0316 - digit2_accuracy: 0.5924 - digit2_loss: 2.6485 - digit3_accuracy: 0.5523 - digit3_loss: 1.4054 - digit4_accuracy: 0.5434 - digit4_loss: 1.3108 - digit5_accuracy: 0.4633 - digit5_loss: 1.6412 - loss: 4.5517

15/15 ━━━━━━━━━━━━━━━━━━━━ 22s 869ms/step - digit1_accuracy: 0.9087 - digit1_loss: 0.0316 - digit2_accuracy: 0.5924 - digit2_loss: 2.6485 - digit3_accuracy: 0.5523 - digit3_loss: 1.4054 - digit4_accuracy: 0.5434 - digit4_loss: 1.3108 - digit5_accuracy: 0.4633 - digit5_loss: 1.6412 - loss: 4.5517 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3259 - val_digit2_accuracy: 0.7458 - val_digit2_loss: 1.1064 - val_digit3_accuracy: 0.7458 - val_digit3_loss: 1.0449 - val_digit4_accuracy: 0.6780 - val_digit4_loss: 1.4360 - val_digit5_accuracy: 0.4576 - val_digit5_loss: 1.4446 - val_loss: 5.5020 - learning_rate: 1.0000e-05
Epoch 2/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 724ms/step - digit1_accuracy: 0.9065 - digit1_loss: 0.0337 - digit2_accuracy: 0.5835 - digit2_loss: 0.6766 - digit3_accuracy: 0.6102 - digit3_loss: 0.8134 - digit4_accuracy: 0.5345 - digit4_loss: 1.1913 - digit5_accuracy: 0.4655 - digit5_loss: 1.6696 - loss: 4.6464 - val_digit1_accuracy: 0.9322 - val_digit1_loss: 0.3483 - val_digit2

In [99]:
model.evaluate(test_ds)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step - digit1_accuracy: 1.0000 - digit1_loss: 0.0638 - digit2_accuracy: 0.9200 - digit2_loss: 0.7521 - digit3_accuracy: 0.7600 - digit3_loss: 1.0822 - digit4_accuracy: 0.7200 - digit4_loss: 1.3679 - digit5_accuracy: 0.5200 - digit5_loss: 1.7646 - loss: 5.1791


[5.179050922393799,
 0.06382554769515991,
 0.7520961165428162,
 1.0822291374206543,
 1.3678734302520752,
 1.7645622491836548,
 1.0,
 0.9200000166893005,
 0.7599999904632568,
 0.7200000286102295,
 0.5199999809265137]

### The metric that actually matters: full 5-digit exact match

Per-digit accuracy is misleading here -- `digit1` is ~99% one class, so any *average* across digits gets inflated by a target that was never hard. The metric that answers "did we read the meter correctly" is: are all 5 digits correct on the same image, at once.

In [100]:
preds = model.predict(test_ds, verbose=0)
# preds is a list of 5 arrays, one per digit, each shape (N, 10)

pred_digits = np.stack([np.argmax(p, axis=1) for p in preds], axis=1)  # (N, 5)

true_digits = []
for batch in test_ds.unbatch():
    image, labels, weights = batch
    true_digits.append([int(labels[i].numpy()) for i in range(5)])
true_digits = np.array(true_digits)  # (N, 5)

per_digit_acc = (pred_digits == true_digits).mean(axis=0)
exact_match_acc = (pred_digits == true_digits).all(axis=1).mean()

for i, acc in enumerate(per_digit_acc, start=1):
    print(f"digit{i} accuracy: {acc:.4f}")
print(f"\nFull 5-digit exact-match accuracy: {exact_match_acc:.4f}")


digit1 accuracy: 1.0000
digit2 accuracy: 0.9200
digit3 accuracy: 0.7600
digit4 accuracy: 0.7200
digit5 accuracy: 0.5200

Full 5-digit exact-match accuracy: 0.3600


In [101]:
IMG_HEIGHT = 96
IMG_WIDTH = 320

def predict_meter(image_path):

    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))

    # Raw [0,255] -- matches training exactly (ResNet50 preprocess_input is
    # applied inside the model itself via the Lambda layer).
    image = tf.cast(image, tf.float32)
    image = tf.expand_dims(image, axis=0)

    preds = model.predict(image, verbose=0)

    digits = []
    for pred in preds:
        digits.append(str(np.argmax(pred[0])))

    return "".join(digits)


### Test-time augmentation (TTA)

Run each image through several mild augmented copies, average the softmax probabilities per digit across them, then take the argmax of the averaged probabilities. This uses a much gentler transform than training augmentation -- large rotations/blur/noise at inference would push the image away from what a real photo looks like, rather than just sampling nearby plausible variations of it.

In [102]:
transform_tta = A.Compose([
    A.Affine(rotate=(-2, 2), translate_percent=(-0.02, 0.02), scale=(0.97, 1.03), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
])


def predict_meter_tta(image_path, n_augments=8):

    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH)).numpy()

    variants = [image]  # always include the un-augmented original
    for _ in range(n_augments - 1):
        img_uint8 = np.clip(image, 0, 255).astype(np.uint8)
        aug = transform_tta(image=img_uint8)["image"]
        variants.append(aug.astype(np.float32))

    batch = np.stack(variants, axis=0)  # (n_augments, H, W, 3)

    preds = model.predict(batch, verbose=0)  # list of 5 arrays, each (n_augments, 10)

    digits = []
    for p in preds:
        avg_probs = p.mean(axis=0)   # average softmax across augmented copies
        digits.append(str(np.argmax(avg_probs)))

    return "".join(digits)


In [135]:
reading = predict_meter(
    "../data_set_generator_for_cnns_only/data_set/test/0642_04481372763crop_0.png"
)

print(reading)

00005
